# THE DISCOMBOBULATOR — FULL PIPELINE
## by Rebel AI

```
  ██████╗ ██╗   ██╗ ██████╗ ██████╗ ███████╗██████╗ 
  ██╔══██╗╚██╗ ██╔╝██╔═══██╗██╔══██╗██╔════╝██╔══██╗
  ██████╔╝ ╚████╔╝ ██║   ██║██████╔╝█████╗  ██████╔╝
  ██╔═══╝   ╚██╔╝  ██║   ██║██╔══██╗██╔══╝  ██╔══██╗
  ██║        ██║   ╚██████╔╝██║  ██║███████╗██║  ██║
  ╚═╝        ╚═╝    ╚═════╝ ╚═╝  ╚═╝╚══════╝╚═╝  ╚═╝
```

> [!IMPORTANT]
> **ONE-CLICK FULL PIPELINE**
> 1. Abliterate a model with OBLITERATUS (GPU)
> 2. Auto-detect the output
> 3. Deploy as OpenAI-compatible API with ngrok tunnel
> 4. Get API key instantly

---

## ⚡ Quick Start

1. **Enable GPU:** Runtime → Change runtime type → **T4 GPU**
2. **Run all cells** (Ctrl+F9)
3. **Select a model** to abliterate when prompted
4. Wait ~15–30 min total (depends on model size)
5. **Copy your API endpoint + key** from the final cell

---

## 🔄 Pipeline Stages

| Stage | What | Time |
|-------|------|------|
| **1. Install** | OBLITERATUS + vLLM + ngrok | 3–5 min |
| **2. Abliterate** | Select model → run `advanced` method | 10–40 min |
| **3. Serve** | vLLM server + ngrok tunnel + key gen | 1–2 min |
| **4. Ready** | API live, test request sent | instant |

---

**☠️  "From guardrails to gateway." — Rebel AI*


In [ ]:
# Kali theme
from IPython.display import HTML, display
kali_theme = """
<style>
  :root{--kali-bg:#0a0a0a;--kali-bg-light:#111;--kali-text:#00ff41;--kali-accent:#00ff41;--kali-orange:#ff6f00;--kali-cyan:#00ffff;--kali-dim:#008f11}
  body,.cell,.output,.input{background:var(--kali-bg)!important;color:var(--kali-text)!important;font-family:'JetBrains Mono','Fira Mono','Consolas',monospace!important}
  .cell .input,.cell .output{border:1px solid var(--kali-dim)!important;border-radius:4px;padding:10px}
  .cell .input{background:var(--kali-bg-light)!important;border-left:4px solid var(--kali-orange)!important}
  .cell .output{background:#050505!important;border-left:4px solid var(--kali-accent)!important}
  h1,h2,h3,h4,h5,h6{color:var(--kali-orange)!important;text-shadow:0 0 5px var(--kali-orange)}
  a{color:var(--kali-cyan)!important;text-decoration:none}a:hover{color:var(--kali-orange)!important}
  code{background:var(--kali-bg-light)!important;color:var(--kali-cyan)!important;border:1px solid var(--kali-dim);padding:2px 6px;border-radius:3px}
  ::-webkit-scrollbar{width:8px;height:8px}::-webkit-scrollbar-track{background:var(--kali-bg)}
  ::-webkit-scrollbar-thumb{background:var(--kali-dim);border-radius:4px}
  .cell{margin-left:12px!important;border-left:3px solid var(--kali-dim)!important;padding-left:15px!important}
  .prompt{color:var(--kali-orange)!important}.output_text{color:var(--kali-text)!important}
</style>
"""
display(HTML(kali_theme))
print("[+] Kali full-pipeline interface: ONLINE\n")


In [ ]:
# [STEP 0] GPU Verification\n!nvidia-smi\nprint("[+] T4 GPU acquired — pipeline initiated.\n")

In [ ]:
# [STEP 1] Install OBLITERATUS + vLLM + ngrok\nprint("[+] Installing full pipeline stack…\n")\n\n# OBLITERATUS (ablitation engine)\n!git clone https://github.com/elder-plinius/OBLITERATUS.git 2>/dev/null || echo "[i] OBLITERATUS repo already exists"\n%cd OBLITERATUS\n!pip install -q --upgrade pip setuptools wheel\n!pip uninstall -y torch torchvision torchaudio -q 2>/dev/null\n!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n!pip install -q -e \"./." --no-cache-dir --break-system-packages\n%cd ..\n\n# API server stack\n!pip install -q "vllm>=0.4.0" pyngrok fastapi uvicorn[standard] --no-cache-dir --break-system-packages\n\nprint("✅ Stack installed — ready to configure.\n")\nimport subprocess, sys, time, os, json, secrets, string, glob\nfrom pathlib import Path

In [ ]:
# [STEP 2] Select model to abliterate\nprint("[+] Loading OBLITERATUS model catalog…\n")\n\n# Get popular models from OBLITERATUS presets\npreset_models = [\n    \"mistralai/Mistral-7B-Instruct-v0.3\",\n    \"microsoft/Phi-3.5-mini-instruct\",\n    \"google/gemma-2-2b-it\",\n    \"meta-llama/Llama-3.1-8B-Instruct\",\n    \"Qwen/Qwen2.5-7B-Instruct\",\n    \"HuggingFaceTB/SmolLM3-3B\",\n    \"TinyLlama/TinyLlama-1.1B-Chat-v1.0\",\n]\n\nprint("Recommended models (fast on T4):"):\nfor i, m in enumerate(preset_models, 1):\n    print(f"  [{i}] {m}")\nprint(f"  [C] Custom model (enter HuggingFace ID)")\n\n# Get user choice\nchoice = input(\"\nSelect model (1-{0} or C): \".format(len(preset_models))).strip()\n\nif choice.upper() == 'C':\n    MODEL_ID = input(\"Enter HuggingFace model ID: \").strip()\nelse:\n    try:\n        idx = int(choice) - 1\n        MODEL_ID = preset_models[idx]\n    except (ValueError, IndexError):\n        print(\"[!] Invalid choice — defaulting to Mistral-7B\")\n        MODEL_ID = \"mistralai/Mistral-7B-Instruct-v0.3\"\n\nprint(f"\n[+] Target model: {MODEL_ID}\")\nprint("[+] This will be saved to ./abliterated-models/\n")

In [ ]:
# [STEP 3] Run OBLITERATUS abliteration\nprint(\\"[+] Starting abliteration… this will take 10–40 minutes.\n\\")\nprint(\\"    Method: advanced (multi-direction SVD, norm-preserving)\\n\\")\nprint(\\"    Output: ./abliterated-models/{MODEL_ID.split('/')[-1]}\\n\\")\n\n# Build command with smart defaults\nabl_cmd = [\n    sys.executable, '-m', 'obliteratus.cli', 'obliterate', MODEL_ID,\n    '--method', 'advanced',\n    '--direction-method', 'diff_means',\n    '--n-directions', '4',\n    '--refinement-passes', '2',\n    '--regularization', '0.1',\n    '--quantization', '4bit',\n    '--output-dir', './abliterated-models',\n    '--verify-sample-size', '10'\n]\n\n# Run ablation (stream output live)\nprint(\\"[+] Launching ablation process…\n\\")\nabl_proc = subprocess.Popen(\n    abl_cmd,\n    stdout=subprocess.PIPE,\n    stderr=subprocess.STDOUT,\n    cwd='/content/OBLITERATUS',\n    universal_newlines=True\n)\n\n# Stream output in real-time\nimport re\nprogress_pattern = re.compile(r'(Progress|Refusal|Perplexity|Step)', re.IGNORECASE)\n\nwhile True:\n    line = abl_proc.stdout.readline()\n    if not line:\n        if abl_proc.poll() is not None:\n            break\n        continue\n    \n    # Only print progress-related lines to reduce noise\n    if progress_pattern.search(line) or \"[+]\" in line or \"✅\" in line or \"[!]\" in line:\n        print(line.rstrip())\n    \n    # Check for completion marker\n    if \"Abliteration complete\" in line or \"Results saved to\" in line or abl_proc.poll() not in [None, 0]:\n        break\n\nexit_code = abl_proc.wait()\nprint(f\\"[+] Abliteration process exited: {exit_code}\\\n\")\nif exit_code != 0:\n    print(\\"[!] Ablation may have failed — check output above.\\n\\")\n    # Still try to find model in output dir\nelse:\n    print(\\"[✓] Abliteration completed successfully!\\n\\")

In [ ]:
# [STEP 4] Auto-detect abliterated model\nprint(\\"[+] Locating abliterated model output…\\n\")\n\ncandidates = []\n# Check common output locations\nfor pattern in [\n    './abliterated-models/*',\n    '/content/OBLITERATUS/abliterated-models/*',\n    '/content/abliterated-models/*',\n]:\n    candidates.extend(glob.glob(pattern, recursive=False))\n\n# Filter to directories only\nmodel_dirs = [d for d in candidates if Path(d).is_dir()]\n\nif not model_dirs:\n    print(\\"[!] Could not find abliterated model directory.\\n\\")\n    print(\\"    Expected: ./abliterated-models/<model_name>\\n\\")\n    print(\\"    Available files:")\n    !find /content -maxdepth 4 -type d -name \"*abliterated*" 2>/dev/null | head -10\n    raise SystemExit(\\"Pipeline failed — model output not detected.\\n\\")\n\nABLITERATED_MODEL = model_dirs[0]\nprint(f\\"[✓] Abliterated model detected: {ABLITERATED_MODEL}\\\n\")\n\n# Show directory contents\nprint(\\"[+] Output contents:")\n!ls -lah {ABLITERATED_MODEL} | head -15

In [ ]:
# [STEP 5] Generate API credentials\nimport secrets, string, json\nprint(\\"[+] Generating API key…\\n\")\n\nAPI_KEY = \"\".join(secrets.choice(string.ascii_letters+string.digits) for _ in range(32))\nAPI_PORT = 8000\n\nconfig = {\n    \"model_path\": ABLITERATED_MODEL,\n    \"model_id\": MODEL_ID,\n    \"api_key\": API_KEY,\n    \"port\": API_PORT,\n    \"ngrok_url\": None,\n    \"timestamp\": __import__('datetime').datetime.now().isoformat()\n}\nPath('/content/server_config.json').write_text(json.dumps(config, indent=2))\nPath('/content/API_KEY.txt').write_text(API_KEY)\n\nprint(f\\"[✓] API Key: {API_KEY}\\\n\")\nprint(f\\"[✓] Model path: {ABLITERATED_MODEL}\\\n\")\nprint(\\"[✓] Config written to /content/server_config.json\\n\")

In [ ]:
# [STEP 6] Launch vLLM API server\nprint(\\"[+] Starting vLLM inference server…\\n\")\nvllm_cmd = [\n    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',\n    '--model', ABLITERATED_MODEL,\n    '--port', str(API_PORT),\n    '--host', '0.0.0.0',\n    '--max-model-len', '4096'\n]\nserver = subprocess.Popen(vllm_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd='/content')\n\nprint(\\"[+] Waiting for server startup…\\n\")\ntime.sleep(12)\n\n# Peek at startup logs\nstarted = False\nfor _ in range(40):\n    line = server.stdout.readline()\n    if line:\n        msg = line.decode('utf-8', errors='replace').rstrip()\n        print(f\\"    {msg}\\\n\")\n        if any(x in msg.lower() for x in ['uvicorn', 'application startup', 'running on']):\n            started = True\n            break\n    time.sleep(1)\n\nif started:\n    print(\\"[✓] vLLM server is UP!\\n\\")\nelse:\n    print(\\"[!] Server may still be initializing… proceeding to tunnel setup.\\n\\")

In [ ]:
# [STEP 7] Local API test\nimport requests\nurl = f\\"http://localhost:{API_PORT}/v1/chat/completions\\"\nheaders = {\\"Authorization\\": f\\"Bearer {API_KEY}\", \"Content-Type\\": \"application/json\\"}\npayload = {\\"model\\": \"ablitated_model\\", \"messages\\": [{\\"role\\": \"user\\", \"content\\": \"Test — you are an abliterated model. Confirm?\\"}], \"max_tokens\\": 64}\n\nprint(\\"[+] Testing local endpoint…\\n\")\ntry:\n    r = requests.post(url, json=payload, headers=headers, timeout=20)\n    print(f\\"[✓] HTTP {r.status_code}\\\n\")\n    resp = r.json()\n    reply = resp.get(\\"choices\\",[{}])[0].get(\\"message\\",{}).get(\\"content\\",\\"[empty]\\")[:250]\n    print(f\\"[REPLY] {reply}\\\n\")\nexcept Exception as e:\n    print(f\\"[!] Local test failed: {e}\\\n\")\n    print(\\"    Server may need more time — check next cell for ngrok URL.\\n\")

In [ ]:
# [STEP 8] Expose via ngrok public tunnel\nprint(\\"[+] Creating ngrok tunnel…\\n\")\nfrom pyngrok import ngrok\n\nNGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')\nif NGROK_TOKEN:\n    ngrok.set_auth_token(NGROK_TOKEN)\n    print(\\"[✓] ngrok auth token configured\\\n\")\nelse:\n    print(\\"[!] No NGROK_TOKEN env — anonymous tunnel (rate-limited)\\\n\")\n\n# Kill old tunnels\nngrok.kill()\ntime.sleep(1)\n\ntunnel = ngrok.connect(API_PORT, \"http\")\nPUBLIC_URL = tunnel.public_url\nconfig[\\"ngrok_url\\"] = PUBLIC_URL\nPath('/content/server_config.json').write_text(json.dumps(config, indent=2))\n\n# Final banner\nbanner = \"=\" * 60\nprint(\\"\n" + banner)\nprint(\\"🔥  DISCOMBOBULATOR PIPELINE COMPLETE\\n")\nprint(\\"=" * 60)\nprint(f\\"\n📡 API Endpoint:\\n   {PUBLIC_URL}/v1/chat/completions\\\n\")\nprint(f\\"🔑 API Key:\\n   {API_KEY}\\\n\")\nprint(f\\"📦 Model: {MODEL_ID}\\\n\")\nprint(\\"=" * 60)\nprint(\\"\n[+] TEST FROM ANYWHERE (replace API_KEY below):\\n\")\nprint(\\"curl -X POST \\\"" + PUBLIC_URL + \"/v1/chat/completions\\" \\\"\\\n\")\nprint(\\"     -H \\\"Authorization: Bearer \\\" + \"" + API_KEY + \"\\" \\\"\\\n\")\nprint(\\"     -H \\\"Content-Type: application/json\\" \\\"\\\n\")\nprint(\\"     -d '{\\\"model\\\":\\\"ablitated_model\\\",\\\"messages\\\":[{\\\"role\\\":\\\"user\\\",\\\"content\\\":\\\"Hello\\\"}]}'\\n\")\nprint(\\"\n⚠️  Colab auto-disconnects ~12h — keep tab open!\\n\")

## ⏹️ Shutdown

**Stop the API server:**
```python
ngrok.disconnect(PUBLIC_URL)
ngrok.kill()
!pkill -f 'vllm.entrypoints.openai.api_server'
import torch; torch.cuda.empty_cache()
```

**Export credentials:**
```python
from google.colab import files
files.download('/content/server_config.json')
files.download('/content/API_KEY.txt')
```

**⚠️  Notes:**
- Keep Colab tab open — closing kills everything
- For 24/7 deployment, move to a cloud VM
- Free ngrok URL changes on restart (get new one each run)

---

**☠️  Rebel AI — Unstoppable Power Level**
